In [5]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()

# Gör så att Python hittar src/
sys.path.insert(0, str(PROJECT_ROOT))

# Importer från projektet
from src.config import load_config
from src.extracting_data import hamta_pm25

# Läs config
CONFIG_PATH = PROJECT_ROOT / "config.toml"
config = load_config(CONFIG_PATH, PROJECT_ROOT)


# Funktion som hittar exakt en matchande fil
def find_csv(directory, pattern):
    files = list(directory.glob(pattern))

    if len(files) != 1:
        raise ValueError(
            f"Förväntade 1 fil för '{pattern}', hittade {len(files)}: {files}"
        )

    return files[0]


# Luftdata / PM2.5
air_path = find_csv(config.raw_dir, "train_*_pm25tidsserie_*.csv")
air_df = pd.read_csv(air_path)

# Väderdata
weather_path = find_csv(config.raw_dir, "train_*_vaderstationid_*.csv")
weather_df = pd.read_csv(weather_path)

# Sammanfogad data (utan tidsförskjutning)
merged_path = find_csv(config.merged_dir, "train_merged_20*.csv")
merged_df = pd.read_csv(merged_path)

# Kontrollera vilka filer som lästs in
# print("PM2.5:", air_path.name)
# print("Väder:", weather_path.name)
# print("Merged:", merged_path.name)

### Finns dubletterna i rådata?

In [2]:
# Konvertera tidsstämplarna till svensk tid
air_df["tid"] = pd.to_datetime(
    air_df["tid"], utc=True
).dt.tz_convert("Europe/Stockholm")

# Kontrollera dubbletter
print("Antal dubbla tidsstämplar:")
print(air_df["tid"].duplicated().sum())

# Visa alla rader som har en dubbel tidsstämpel
duplicates = air_df[
    air_df["tid"].duplicated(keep=False)
].sort_values("tid")

display(duplicates)

# Gruppera alla dubblerade tidsstämplar
duplicate_groups = duplicates.groupby("tid")["pm25"]

print("Antal unika dubblerade tidsstämplar:", duplicate_groups.ngroups)

print(
    "Alla dubbletter vid månadsskiften:",
    ((duplicates["tid"].dt.day == 1) &
     (duplicates["tid"].dt.hour == 0)).all()
)

print(
    "Alla dubblerade värden identiska:",
    duplicate_groups.nunique().eq(1).all()
)

Antal dubbla tidsstämplar:
31


,tid,pm25
142,2023-06-01 00:00:00+02:00,6.857
143,2023-06-01 00:00:00+02:00,6.857
863,2023-07-01 00:00:00+02:00,3.727
864,2023-07-01 00:00:00+02:00,3.727
1608,2023-08-01 00:00:00+02:00,2.300
...,...,...
20643,2025-10-01 00:00:00+02:00,2.731
21388,2025-11-01 00:00:00+01:00,8.742
21389,2025-11-01 00:00:00+01:00,8.742
22109,2025-12-01 00:00:00+01:00,4.462


Antal unika dubblerade tidsstämplar: 31
Alla dubbletter vid månadsskiften: True
Alla dubblerade värden identiska: True


### Test: Dubbletter vid månadsskiften

Testar hamta_pm25() över månadsskiftet juni–juli 2025 för att kontrollera att överlappande observationer tas bort.

**Resultat:** API:et returnerade 74 observationer. Efter rensning återstod 73, utan dubbla tidsstämplar. Ändringen fungerar som avsett.

In [3]:
# Testa datahämtningen över ett månadsskifte
test_start = "2025-06-30"
test_slut = "2025-07-02"

# Använd en sökväg som inte finns så att API:et anropas
test_path = config.raw_dir / "temp_pm25_test.csv"

assert not test_path.exists(), "Testfilen finns redan!"

test_df = hamta_pm25(
    test_start,
    test_slut,
    6247,
    "https://datavardluft.smhi.se/52North/api",
    test_path
)

print("Antal observationer:", len(test_df))
print("Antal dubbla tidsstämplar:", test_df.index.duplicated().sum())

display(test_df.head())

Hämtar 2025-06...
Hämtar 2025-07...
Antal observationer: 72
Antal dubbla tidsstämplar: 0


,pm25
tid,
2025-06-30 00:00:00+02:00,9.875
2025-06-30 01:00:00+02:00,7.335
2025-06-30 02:00:00+02:00,7.127
2025-06-30 03:00:00+02:00,7.340
2025-06-30 04:00:00+02:00,7.475


### Test: Tidsintervallets slutdatum
Kontrollerar att hamta_pm25() endast returnerar observationer inom det begärda tidsintervallet.
API:et returnerade en extra observation den 1 januari 2026. Efter filtrering är sista observationen korrekt: 31 december 2025 kl. 23.00.

In [4]:
# Testa hämtning över årsskiftet
test_start = "2025-12-30"
test_slut = "2025-12-31"

test_df = hamta_pm25(
    test_start,
    test_slut,
    6247,
    "https://datavardluft.smhi.se/52North/api",
    config.raw_dir / "temp_pm25_test.csv"
)

# Kontrollera tidsintervallet
print("Första observation:", test_df.index.min())
print("Sista observation:", test_df.index.max())

# Visa observationer efter slutdatumet
slutgrans = pd.Timestamp("2026-01-01", tz="Europe/Stockholm")

display(test_df[test_df.index >= slutgrans])

Hämtar 2025-12...
Första observation: 2025-12-30 00:00:00+01:00
Sista observation: 2025-12-31 23:00:00+01:00


,pm25
tid,


In [ ]:
# Kontrollera den nya PM2.5-filen
air_df["tid"] = pd.to_datetime(
    air_df["tid"], utc=True
).dt.tz_convert("Europe/Stockholm")

start = pd.Timestamp("2023-05-26", tz="Europe/Stockholm")
slut = pd.Timestamp("2026-01-01", tz="Europe/Stockholm")

print("\nPM2.5 data:")
print("Antal observationer:", len(air_df))
print("Antal dubbletter:", air_df["tid"].duplicated().sum())
print("Första observation:", air_df["tid"].min())
print("Sista observation:", air_df["tid"].max())

print(
    "Observationer utanför tidsintervallet:",
    ((air_df["tid"] < start) | (air_df["tid"] >= slut)).sum()
)


merged_df["tid"] = pd.to_datetime(
    merged_df["tid"], utc=True
).dt.tz_convert("Europe/Stockholm")

print("\nMerged data:")
print("Antal observationer:", len(merged_df))
print("Antal dubbletter:", merged_df["tid"].duplicated().sum())
print("Första observation:", merged_df["tid"].min())
print("Sista observation:", merged_df["tid"].max())

print(
    "Observationer utanför tidsintervallet:",
    ((merged_df["tid"] < start) | (merged_df["tid"] >= slut)).sum()
)


PM2,5 data:
Antal observationer: 22823
Antal dubbletter: 0
Första observation: 2023-05-26 02:00:00+02:00
Sista observation: 2025-12-31 23:00:00+01:00
Observationer utanför tidsintervallet: 0

Merged data:
Antal observationer: 22825
Antal dubbletter: 0
Första observation: 2023-05-26 00:00:00+02:00
Sista observation: 2025-12-31 23:00:00+01:00
Observationer utanför tidsintervallet: 0
